# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed-Qasim608/Flyrank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task

This is a **ranking problem**.

The goal is to rank content pages by their likelihood of needing a refresh, so teams can review the highest-priority pages first.

Ranking fits better than simple classification because the output is not only "needs refresh / does not need refresh"; the business goal is to create an ordered list of pages based on urgency and impact.

In [ ]:
# No model is required yet.
# This cell checks the available data for ranking signals.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Number of pages:", len(df))
print("Available columns:")
print(df.columns.tolist())

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

The target is a **proxy label** based on observed search performance changes.

A page is considered declining when its recent impressions decrease compared to its previous period. This label comes from measured search performance data, not from a manual decision.

Example:
- Declining = recent impressions are significantly lower than the previous period.
- Stable = no major decline observed.

This does not prove why a page declined; it only identifies pages showing decline signals.

In [ ]:
# Create the proxy target from observed trend direction

df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Total pages:", len(df))
print("Declining pages:", df["is_declining"].sum())
print("Declining rate:", round(df["is_declining"].mean(), 3))

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric

The main metric will be **Precision@50**.

Precision@50 measures how many of the top 50 recommended pages are actually declining pages.

A higher Precision@50 means the model is better at helping teams find useful pages to review first.

This metric matches the business goal because the team has limited time and needs the highest-value pages at the top of the list.

In [ ]:
# Example calculation of the baseline Precision@50

def precision_at_k(scores, labels, k):
    order = scores.sort_values(ascending=False).index[:k]
    return labels.loc[order].mean()

# Simple baseline score using impressions
baseline_score = df["impressions_90d"]

print(
    "Baseline Precision@50:",
    round(
        precision_at_k(
            baseline_score,
            df["is_declining"],
            50
        ),
        3
    )
)

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

## Unit of analysis

The unit of analysis is **one content page**.

Each dataframe row represents one page with its observed search performance signals, such as impressions, clicks, ranking position, and content-related features.

The model will learn patterns across pages to prioritize which pages need review.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML beats a fixed rule

A fixed rule such as "refresh pages older than 180 days with high impressions" only captures one simple pattern.

ML can combine multiple signals together, such as impressions change, ranking position, click-through rate, content age, and query coverage.

The relationship between these signals is complex, so a learned model can identify patterns that are difficult to describe with a single if/else rule.

The result is a more flexible decision-support system rather than a manually created score.

In [ ]:
# Check available signals that could be used by an ML model

possible_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]

available = [col for col in possible_features if col in df.columns]

print("Features available for modeling:")
print(available)

df[available].describe()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.